# Multisine Signal Tone and Power Harvesting Measurements across Gain and Distance Settings Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and analyzing the "Multisine Signal Tone and Power Harvesting Measurements across Gain and Distance Settings" dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema available at:

**https://sen.science/doi/10.71728/senscience.gk8g-g8p8/fair2.json**

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. Retrieve high-level info from the metadata object.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# URL to the dataset Croissant schema
croissant_url = 'https://sen.science/doi/10.71728/senscience.gk8g-g8p8/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print main dataset information
print(f"Name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")

## 2. Data Overview
Let's review available record sets, fields, and their `@id` attributes. These `@id`s are essential when referencing parts of the dataset with `mlcroissant` functions.

In [ ]:
# List all available record sets and their fields with @id references
record_sets = list(dataset.record_sets)
print(f"Available Record Sets ({len(record_sets)}):\n")
for recset in record_sets:
    print(f"- RecordSet name: {recset.name}  |  @id: {recset.id}")
    print("  Fields:")
    for field in recset.fields:
        print(f"    - {field.name}  |  @id: {field.id}  |  type: {field.data_type}")
    print()
# Save the first record set @id for later use
if record_sets:
    main_record_set_id = record_sets[0].id
    print(f"Main record set id: {main_record_set_id}")
else:
    main_record_set_id = None

## 3. Data Extraction
Load data from each record set into separate DataFrames. We use the record set and field `@id` values identified above.

> **Note:** If you wish to extract a subset, modify `record_set_ids` accordingly.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for rec_id in record_set_ids:
    records = list(dataset.records(record_set=rec_id))
    df = pd.DataFrame(records)
    dataframes[rec_id] = df
    print(f"\nRecordSet @id: {rec_id}")
    print(f"Columns: {df.columns.tolist()}")

# Display the first few rows if at least one record set was loaded
if record_set_ids:
    main_id = record_set_ids[0]
    print(dataframes[main_id].head())
else:
    print("No record sets are defined in the dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data preparation steps. We'll: 
- Select a numeric field for analysis.
- Filter records based on a value threshold.
- Normalize the selected numeric field.
- Optionally, group by an experimental configuration field.

Field references are always made using their `@id`.

In [ ]:
# Fill in the correct record set and field @ids based on the previous overview.
# For demonstration, we infer typical field names: e.g. 'Power_mW', 'gain', 'distance', 'tones', 'partition'.
# Update below as needed to match actual IDs from output of the previous cell!

main_rec_id = main_id  # Use main record set
main_df = dataframes[main_rec_id]

# Try to auto-detect a numeric field. Adjust these as appropriate (make sure they appear in the DataFrame columns).
possible_numeric_fields = ['Power_mW', 'power', 'power_mw', 'rssi', 'harvested_power']
numeric_field = None
for col in main_df.columns:
    for candidate in possible_numeric_fields:
        if candidate.lower() in col.lower():
            numeric_field = col
            break
    if numeric_field:
        break
if not numeric_field:
    # Fallback: use the first float column
    for col in main_df.columns:
        if pd.api.types.is_numeric_dtype(main_df[col]):
            numeric_field = col
            break

if not numeric_field:
    raise ValueError("No numeric field detected. Please update the field selection cell with the appropriate @id.")

print(f"Using numeric field: '{numeric_field}' (@id)")

# Simple threshold (mean + std for demonstration)
threshold = main_df[numeric_field].mean() + main_df[numeric_field].std()
filtered_df = main_df[main_df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.3f}:")
print(filtered_df.head())

# Normalize
normalized_col = numeric_field + '_normalized'
filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, normalized_col]].head())

# Try grouping by a likely categorical/experiment configuration field
possible_group_fields = ['tones', 'gain', 'distance', 'partition']
group_field = None
for col in main_df.columns:
    for candidate in possible_group_fields:
        if candidate.lower() in col.lower():
            group_field = col
            break
    if group_field:
        break
if group_field and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped data by '{group_field}':")
    print(grouped_df.head())
else:
    print("\nNo suitable group field detected.")

## 5. Visualization
Visualize the distribution of the chosen numeric field and, if available, mean values grouped by the experimental field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(main_df[numeric_field], bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.show()

# If grouped_df exists, plot barplot
if 'grouped_df' in locals():
    plt.figure(figsize=(8,4))
    sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
    plt.title(f"Mean {numeric_field} grouped by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field}")
    plt.show()
else:
    print("No grouped data to plot.")

## 6. Conclusion

- The dataset offers high-quality measurements for the RF energy harvesting/trials domain, configurable by gain, distance, and number of tones.
- Using `mlcroissant`, we explored record sets, loaded all records, filtered by power measurements, normalized values, and visualized their distributions.
- Field and record set `@id`s ensure robust, reproducible references in your code.

**Next steps**: Dive deeper (e.g., fit regression models, classify experiments, or analyze hardware effects) based on your use-case!
